# Chuỗi tăng 3.4% nhưng 33 cửa hàng giảm hơn 10%: vì sao, và làm gì?

Bộ `store-sales` (Rossmann): `train.csv` (doanh số ngày, 1 115 cửa hàng, 01/01/2013 tới 31/07/2015) và `store.csv`
(thông tin cửa hàng: `StoreType`, `Assortment`, `CompetitionDistance` (mét tới đối thủ gần nhất),
`CompetitionOpenSinceMonth`/`Year` (đối thủ đó mở từ khi nào), `Promo2` (chương trình coupon dài hạn) ...).

**Bối cảnh.** Head of Retail Operations nhận báo cáo nửa năm: doanh số 6 tháng đầu 2015 của toàn chuỗi tăng 3.4% so
với cùng kỳ 2014, nhưng 33 cửa hàng giảm hơn 10%. Bà hỏi: **"Vì sao các cửa hàng này giảm? Cửa hàng nào cần can
thiệp, can thiệp thế nào? Tôi cần trả lời trong buổi họp vùng tuần sau."**

Bạn đi một vòng điều tra root cause: kiểm con số có so được không, phân rã thay đổi của từng cửa hàng (số ngày mở
cửa, lượt khách, chi tiêu mỗi khách), tìm nguyên nhân chung (đối thủ mới, đóng cửa sửa chữa, ...), so với nhóm đối
chứng, rồi viết memo.

**Cách làm bài**
- Cell chấm điểm chỉ định nghĩa hàm. Điền vào chỗ `# TODO`; hàm không được sửa dữ liệu đầu vào.
- Cell gắn tag `explore` dùng cho điều tra tự do; khi chấm chúng bị bỏ qua.
- Viết `MEMO.md` (cùng thư mục với notebook) theo khung có sẵn.
- Chấm phần tự động: `uv run pytest tracks/data-analyst/modules/13-business-financial-analytics/exercises/advanced/01-store-decline-investigation -q`. Phần memo chấm theo rubric trong lời giải.

In [ ]:
# explore: tìm gốc repo, tải bộ dữ liệu nếu chưa có, đọc hai tệp
import sys
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
sys.path.insert(0, str(ROOT))
from data.download_dataset import ensure
import pandas as pd

DATA = ensure("store-sales")
train = pd.read_csv(DATA / "train.csv", parse_dates=["Date"], dtype={"StateHoliday": str})
stores = pd.read_csv(DATA / "store.csv")
print(train.shape, stores.shape)
stores.head()

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

P0 = ("2014-01-01", "2014-06-30")
P1 = ("2015-01-01", "2015-06-30")

## 1. Bảng so sánh từng cửa hàng

`store_yoy(train, p0, p1) -> pd.DataFrame`: `p0`, `p1` là cặp ngày (bắt đầu, kết thúc), **tính cả hai mút**. Mỗi
cửa hàng có dòng trong cả hai kỳ một dòng (index `Store` tăng dần), cột theo thứ tự:

| Cột | Định nghĩa |
| --- | --- |
| `sales0`, `sales1` | tổng `Sales` của kỳ |
| `open_days0`, `open_days1` | tổng `Open` của kỳ (số ngày mở cửa) |
| `customers0`, `customers1` | tổng `Customers` của kỳ |
| `growth` | `sales1 / sales0 − 1` |
| `days_growth` | `open_days1 / open_days0 − 1` |
| `traffic_growth` | lượt khách mỗi ngày mở cửa: `(customers1 / open_days1) / (customers0 / open_days0) − 1` |
| `spc_growth` | chi tiêu mỗi khách: `(sales1 / customers1) / (sales0 / customers0) − 1` |

Với ba thành phần này, `1 + growth = (1 + days_growth)(1 + traffic_growth)(1 + spc_growth)` đúng tuyệt đối.

In [ ]:
def store_yoy(train: pd.DataFrame, p0: tuple, p1: tuple) -> pd.DataFrame:
    """So sánh hai kỳ theo cửa hàng: tổng, tăng trưởng và ba thành phần."""
    # TODO
    raise NotImplementedError

## 2. Đối thủ mở khi nào so với hai kỳ

`competition_timing(stores, p0, p1) -> pd.Series` (index `Store`, tên `competition`): ngày đối thủ mở là ngày 1 của
tháng `CompetitionOpenSinceMonth`, năm `CompetitionOpenSinceYear`. Thiếu một trong hai cột thì `"unknown"`. Còn lại:

| Nhãn | Ngày đối thủ mở |
| --- | --- |
| `before_p0` | trước ngày đầu `p0` |
| `during_p0` | trong `p0` (tính cả hai mút) |
| `between` | sau `p0`, trước ngày đầu `p1` |
| `during_p1` | trong `p1` |
| `after_p1` | sau `p1` |

`group_growth(yoy, labels) -> pd.DataFrame`: nhóm cửa hàng theo `labels` (Series cùng index `Store`), mỗi nhóm một
dòng: `stores` (số cửa hàng), `sales0`, `sales1`, `growth` (tăng trưởng **gộp** của nhóm = tổng sales1 / tổng
sales0 − 1), `median_growth` (median `growth` của từng cửa hàng), `share_below_10` (tỉ lệ cửa hàng `growth < −0.10`).
Index tên như `labels.name`, sắp theo `growth` tăng dần.

In [ ]:
def competition_timing(stores: pd.DataFrame, p0: tuple, p1: tuple) -> pd.Series:
    """Nhãn thời điểm đối thủ mở so với hai kỳ; tên "competition"."""
    # TODO
    raise NotImplementedError


def group_growth(yoy: pd.DataFrame, labels: pd.Series) -> pd.DataFrame:
    """Tăng trưởng gộp, median và tỉ lệ cửa hàng giảm > 10% theo nhóm."""
    # TODO
    raise NotImplementedError

In [ ]:
# explore: con số của báo cáo - chuỗi và danh sách cửa hàng giảm > 10%
yoy = store_yoy(train, P0, P1)
chain = yoy.sales1.sum() / yoy.sales0.sum() - 1
decliners = yoy[yoy.growth < -0.10].sort_values("growth")
print(f"{len(yoy)} cửa hàng; chuỗi {chain:+.2%}; {len(decliners)} cửa hàng giảm > 10%")
decliners.round(3)

In [ ]:
# explore: TODO - thành phần nào giảm ở nhóm giảm > 10%? (ngày mở cửa, lượt khách, chi tiêu mỗi khách)

In [ ]:
# explore: TODO - đối thủ mới: dùng competition_timing và group_growth

In [ ]:
# explore: khoảng cách tới đối thủ mới mở - gần thì giảm nhiều hơn?
info = stores.set_index("Store")
new_comp = yoy.join(timing).join(info.CompetitionDistance)
new_comp = new_comp[new_comp.competition.isin(["during_p0", "between", "during_p1"])]
new_comp["dist_band"] = pd.cut(new_comp.CompetitionDistance, [0, 500, 2000, np.inf], labels=["<500m", "500m-2km", ">2km"])
print(new_comp.groupby("dist_band", observed=True).growth.agg(["size", "median"]).round(3))

In [ ]:
# explore: cửa hàng giảm nhiều vì đóng cửa? - ngày đóng liên tục trong kỳ 2015
closed = train[(train.Date >= P1[0]) & (train.Date <= P1[1]) & (train.Open == 0) & (train.DayOfWeek < 7)]
runs = closed.groupby("Store").Date.agg(["count", "min", "max"])
print(runs[runs["count"] >= 10].join(yoy.growth).sort_values("count", ascending=False).head(10))

In [ ]:
# explore: các cửa hàng giảm > 10% không có lời giải thích dữ liệu - xem theo tháng
unexplained = decliners.join(timing)
unexplained = unexplained[unexplained.competition.isin(["before_p0", "unknown", "after_p1"]) & (unexplained.days_growth >= 0)]
print(unexplained[["growth", "traffic_growth", "spc_growth", "competition"]].round(3))
month = train[train.Store.isin(unexplained.index[:4])].assign(m=lambda d: d.Date.dt.to_period("M"))
print(month.groupby(["Store", "m"]).Sales.sum().unstack(0).loc["2014-01":"2015-07"].round(-3))

In [ ]:
# explore: TODO - 2-3 chart cho memo, tiêu đề là kết luận có con số

## 3. Viết memo

Mở `MEMO.md`, viết theo khung (200-600 từ, dòng đầu là headline có con số). Người đọc là Head of Retail Operations:
bà cần biết nhóm nào giảm vì lý do gì, nhóm nào cần ai làm gì, và phần nào dữ liệu không trả lời được.